# 扩展主题：RLHF 对齐原理

> **性质**：📖 轻量原理  ｜  **依赖**：不依赖代码，随时可学

## 一句话

让 LLM 的回答**符合人类偏好**（有用、无害、诚实）。ChatGPT 的核心技术，比 ch07 的 DPO 更完整但也更复杂。

## RLHF 三阶段

```
1. SFT（监督微调）    → ch07 已学，让模型学会遵循指令
2. RM（奖励模型）     → 训练一个能给回答打分的模型
3. PPO（强化学习）    → 用 RM 的分数当奖励，优化策略模型
```

> ch07 的 DPO 跳过了第 2、3 步，直接用偏好数据优化。RLHF 更完整但工程复杂。

## 阶段 2：奖励模型（Reward Model）

把分类头改成**打分头**（输出标量分数）。在偏好数据上训练：
- 输入 prompt + response，输出一个分数
- 损失：让 chosen(好回答) 的分数高于 rejected(坏回答)

$$\mathcal{L}_{RM} = -\log\sigma(r(x,y_c) - r(x,y_r))$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 奖励模型 demo：一个给回答打分的网络
class RewardModel(nn.Module):
    """把文本特征映射成标量分数（越高越好）。"""
    def __init__(self, vocab_size=1000, emb_dim=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.score = nn.Linear(emb_dim, 1)   # 打分头（标量输出）
    def forward(self, token_ids):
        # 取最后一个 token 的 embedding 作为文本特征（简化）
        feat = self.emb(token_ids).mean(dim=1)   # [b, emb_dim]
        return self.score(feat).squeeze(-1)      # [b] 分数

def reward_loss(chosen_scores, rejected_scores):
    """奖励模型损失：让好回答分数高于坏回答。"""
    return -F.logsigmoid(chosen_scores - rejected_scores).mean()

# 训练奖励模型
torch.manual_seed(0)
rm = RewardModel()
opt = torch.optim.AdamW(rm.parameters(), lr=0.01)

# 假数据：chosen 和 rejected 的 token 序列
chosen = torch.randint(0, 1000, (8, 10))
rejected = torch.randint(0, 1000, (8, 10))

print("训练奖励模型（让好回答分数 > 坏回答）：")
for epoch in range(30):
    opt.zero_grad()
    c_scores = rm(chosen)
    r_scores = rm(rejected)
    loss = reward_loss(c_scores, r_scores)
    loss.backward(); opt.step()
    if epoch % 10 == 0 or epoch == 29:
        margin = (c_scores.mean() - r_scores.mean()).item()
        print(f"  epoch {epoch}: loss {loss.item():.4f}, 分数差 {margin:+.3f}")
print("\n✓ 奖励模型学会给好回答更高分。")

## 阶段 3：PPO 简化原理

用奖励模型的分数当奖励，通过强化学习优化策略（LLM）。简化版用**策略梯度**展示核心思路：

$$\nabla J = \mathbb{E}\left[ r(x,y) \cdot \nabla\log\pi(y|x) \right]$$

直觉：奖励高的回答，增大其生成概率；奖励低的，减小。

In [ ]:
# PPO/策略梯度 简化 demo（展示「最大化奖励」的优化方向）
# 真实 PPO 还要加 KL 惩罚（防止策略偏离 SFT 模型太远）

# 简化：一个「策略」输出动作概率，用奖励信号优化
torch.manual_seed(0)
policy = nn.Linear(8, 3)   # 输入状态，输出 3 个动作的 logits
opt = torch.optim.AdamW(policy.parameters(), lr=0.05)

def fake_reward(action_idx):
    """假装动作 2 是好动作，奖励高。"""
    rewards = torch.tensor([0.1, 0.3, 1.0])  # 动作 2 奖励最高
    return rewards[action_idx]

print("策略梯度 demo（让策略学会选高奖励动作）：")
for epoch in range(50):
    state = torch.randn(4, 8)              # 4 个状态
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    # 采样动作
    action = torch.multinomial(probs, 1).squeeze(-1)
    reward = fake_reward(action)
    # 策略梯度：reward × log π(a|s)
    log_prob = F.log_softmax(logits, dim=-1).gather(1, action.unsqueeze(1)).squeeze(1)
    loss = -(log_prob * reward).mean()     # 负号：要最大化
    opt.zero_grad()
    loss.backward(); opt.step()
    if epoch % 10 == 0 or epoch == 49:
        with torch.no_grad():
            avg_action = action.float().mean().item()
        print(f"  epoch {epoch}: 平均动作 {avg_action:.2f}（趋近 2=高奖励动作）")
print("\n✓ 策略逐渐偏向高奖励动作，这就是 RLHF 优化 LLM 的核心思路。")

## RLHF vs DPO 对比

| | RLHF | DPO (ch07) |
|---|---|---|
| 阶段 | SFT → RM → PPO | SFT → 直接优化 |
| 需要奖励模型？ | ✅ 要单独训练 | ❌ 不需要 |
| 用强化学习？ | ✅ PPO | ❌ 纯监督损失 |
| 复杂度 | 高（4 个模型） | 低（2 个模型） |
| 效果 | 理论上限高 | 接近 RLHF，工程友好 |

> DPO 用数学推导证明了：偏好数据可以直接优化策略，跳过 RM 和 PPO。这是 2023 年的重要突破。

---
> **小结**：RLHF = SFT + 奖励模型 + PPO，完整但复杂；DPO 是简化替代。
> 本附录聚焦原理理解，真实 RLHF 实现用 `trl` 库的 `PPOTrainer`。